# Sentiment & Emotion Analysis of Netflix Descriptions

This notebook performs **lexicon-based sentiment and emotion analysis** on Netflix movie and TV show descriptions.

We use two datasets:
- **`netflix_titles.csv`** — metadata for ~8,800 Netflix titles including a short description of each
- **`emotion_lexicon.csv`** — the [NRC Emotion Lexicon](https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm), which maps words to eight basic emotions (Plutchik's wheel) and three VAD dimensions (Valence, Arousal, Dominance)

The approach is simple: for each title, we extract words from the description that appear in the lexicon, sum their emotion scores, and attach the result back to the Netflix dataframe. This lets us ask questions like *"Which emotions dominate Netflix descriptions?"* and *"Has the emotional tone of content changed over the years?"*

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

## 2. Load Data

We load both CSVs and take a first look at their structure.

In [ ]:
netflix = pd.read_csv('netflix_titles.csv')
print(f"Netflix titles: {netflix.shape[0]:,} rows, {netflix.shape[1]} columns")
netflix.head(2)

In [ ]:
emotion_lexicon = pd.read_csv('emotion_lexicon.csv')
print(f"Emotion lexicon: {emotion_lexicon.shape[0]:,} words, {emotion_lexicon.shape[1]} columns")
emotion_lexicon.head(2)

### About the Emotion Lexicon

The NRC lexicon provides two scoring systems for each word:

| System | Columns | What it measures |
|--------|---------|------------------|
| **NRC EmoLex** | `anger`, `anticipation`, `disgust`, `fear`, `joy`, `sadness`, `surprise`, `trust` | Binary/continuous scores for Plutchik's 8 basic emotions, plus `positive`/`negative` sentiment |
| **NRC VAD** | `vad_valence`, `vad_arousal`, `vad_dominance` | Continuous scores for the Valence–Arousal–Dominance model |

A word like *"happy"* scores high on `joy`, `trust`, and `positive`, and has a high `vad_valence`.

## 3. Extract Emotion Words from Descriptions

For each Netflix description we:
1. Lowercase and split the text into tokens
2. Strip punctuation from each token
3. Keep only the tokens that appear in the lexicon

The result is a list of matched emotion words per title, stored in a new column `emotion_words`.

In [ ]:
lexicon_word_set = set(emotion_lexicon['word'].str.lower())

def extract_emotion_words(description):
    """Return all words from `description` that appear in the emotion lexicon."""
    if pd.isna(description):
        return []
    tokens = description.lower().split()
    tokens = [t.strip('.,!?;:\'"()[]') for t in tokens]
    return [t for t in tokens if t in lexicon_word_set]

netflix['emotion_words'] = netflix['description'].apply(extract_emotion_words)
netflix['emotion_word_count'] = netflix['emotion_words'].apply(len)

print(f"Average emotion words per description: {netflix['emotion_word_count'].mean():.1f}")
print(f"Titles with at least one emotion word: {(netflix['emotion_word_count'] > 0).sum():,}")

## 4. Score Each Title

For each title we look up all its emotion words in the lexicon and **sum** their scores across every emotion column. This gives us a raw emotional intensity profile per title — the more emotionally loaded the words in a description, the higher the total score.

In [ ]:
emotion_cols_raw = [c for c in emotion_lexicon.columns if c != 'word']

def score_description(word_list):
    """Sum lexicon scores for all words in `word_list`."""
    matched = emotion_lexicon[emotion_lexicon['word'].isin(word_list)]
    if matched.empty:
        return pd.Series(0, index=emotion_cols_raw)
    return matched[emotion_cols_raw].sum()

emotion_scores = netflix['emotion_words'].apply(score_description)
netflix = pd.concat([netflix, emotion_scores], axis=1)

print(f"Columns after scoring: {netflix.shape[1]}")

### Rename Columns for Clarity

We simplify the verbose NRC column names so the dataframe is easier to work with.

In [ ]:
netflix = netflix.rename(columns={
    # EmoLex — eight basic emotions
    'NRC_EmoLex_anger':        'anger',
    'NRC_EmoLex_anticipation': 'anticipation',
    'NRC_EmoLex_disgust':      'disgust',
    'NRC_EmoLex_fear':         'fear',
    'NRC_EmoLex_joy':          'joy',
    'NRC_EmoLex_sadness':      'sadness',
    'NRC_EmoLex_surprise':     'surprise',
    'NRC_EmoLex_trust':        'trust',
    # EmoLex — sentiment polarity
    'NRC_EmoLex_negative':     'sentiment_negative',
    'NRC_EmoLex_positive':     'sentiment_positive',
    # VAD dimensions
    'NRC_VAD_valence':         'vad_valence',
    'NRC_VAD_valence_high':    'vad_valence_high',
    'NRC_VAD_valence_low':     'vad_valence_low',
    'NRC_VAD_valence_polar':   'vad_valence_polar',
    'NRC_VAD_arousal':         'vad_arousal',
    'NRC_VAD_arousal_high':    'vad_arousal_high',
    'NRC_VAD_arousal_low':     'vad_arousal_low',
    'NRC_VAD_arousal_polar':   'vad_arousal_polar',
    'NRC_VAD_dominance':       'vad_dominance',
    'NRC_VAD_dominance_high':  'vad_dominance_high',
    'NRC_VAD_dominance_low':   'vad_dominance_low',
    'NRC_VAD_dominance_polar': 'vad_dominance_polar',
})

# Drop the intermediate word-list column
netflix = netflix.drop(columns=['emotion_words'])

netflix.head(2)

## 5. Overall Emotion Distribution

Which of Plutchik's eight emotions appears most often across all Netflix descriptions?

We sum each emotion column across the entire dataset and rank them.

In [ ]:
EMOTION_COLS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']

EMOTION_COLORS = {
    'anger':        '#e63946',
    'anticipation': '#f4a261',
    'disgust':      '#6a4c93',
    'fear':         '#1d3557',
    'joy':          '#ffbe0b',
    'sadness':      '#457b9d',
    'surprise':     '#fb5607',
    'trust':        '#2a9d8f',
}

emotion_totals = (
    netflix[EMOTION_COLS]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'index': 'emotion', 0: 'score'})
)
# pandas version-safe rename
emotion_totals.columns = ['emotion', 'score']

fig = px.bar(
    emotion_totals,
    x='emotion', y='score',
    color='emotion',
    color_discrete_map=EMOTION_COLORS,
    text='score',
    title='Total Emotion Scores Across All Netflix Descriptions',
    labels={'score': 'Total Score', 'emotion': 'Emotion'},
)
fig.update_traces(texttemplate='%{text:.0f}', textposition='outside')
fig.update_layout(showlegend=False, xaxis_title='Emotion', yaxis_title='Total Score')
fig.show()

## 6. Emotion Trends Over Time

Do Netflix descriptions feel more fearful, joyful, or angry in recent years compared to older titles?

We group by `release_year` (2010–2021, where coverage is densest) and plot the summed emotion scores as a line chart. Because more titles were added in later years, higher absolute scores partly reflect a larger catalogue — but the relative ranking between emotions is still informative.

In [ ]:
emotions_by_year = (
    netflix[
        netflix['release_year'].between(2010, 2021)
    ]
    .groupby('release_year')[EMOTION_COLS]
    .sum()
    .reset_index()
)

fig = px.line(
    emotions_by_year,
    x='release_year',
    y=EMOTION_COLS,
    color_discrete_map=EMOTION_COLORS,
    markers=True,
    title='Emotion Scores by Release Year (2010–2021)',
    labels={'value': 'Total Score', 'release_year': 'Year', 'variable': 'Emotion'},
)
fig.update_layout(
    hovermode='x unified',
    legend_title='Emotion',
    xaxis_title='Release Year',
    yaxis_title='Total Score',
)
fig.show()

## 7. Animated Bar Chart — Emotions by Year

The same data as above, but presented as an **animated bar chart** you can step through or play. This makes it easy to see how the relative height of each emotion bar shifts from year to year.

Press ▶ to animate, or drag the slider to a specific year.

In [ ]:
EMOTION_ORDER = ['anger', 'disgust', 'fear', 'sadness', 'surprise', 'anticipation', 'trust', 'joy']

emotions_melted = emotions_by_year.melt(
    id_vars='release_year',
    value_vars=EMOTION_COLS,
    var_name='emotion',
    value_name='score',
)

fig = px.bar(
    emotions_melted,
    x='emotion', y='score',
    color='emotion',
    animation_frame='release_year',
    color_discrete_map=EMOTION_COLORS,
    category_orders={'emotion': EMOTION_ORDER},
    range_y=[0, emotions_melted['score'].max() * 1.1],
    title='Emotions in Netflix Descriptions by Year (2010–2021)',
    labels={'score': 'Total Score', 'emotion': 'Emotion'},
)
fig.update_layout(
    showlegend=False,
    xaxis_title='Emotion',
    yaxis_title='Total Score',
)
# Slow down the animation for readability
fig.layout.updatemenus[0].buttons[0].args[1].update({
    'frame': {'duration': 1500, 'redraw': True},
    'transition': {'duration': 400},
    'fromcurrent': True,
    'loop': True,
})
fig.show()

## 8. Export Scored Dataset

Save the enriched dataframe — original Netflix metadata plus all emotion scores — as a CSV for downstream use.

In [ ]:
output_cols = [
    'show_id', 'type', 'title', 'director', 'country',
    'date_added', 'release_year', 'rating', 'duration', 'listed_in',
    'emotion_word_count',
    *EMOTION_COLS,
    'sentiment_positive', 'sentiment_negative',
    'vad_valence', 'vad_arousal', 'vad_dominance',
]

netflix[output_cols].to_csv('netflix_with_emotions.csv', index=False)
print("Saved: netflix_with_emotions.csv")